In [ ]:
REXVQA_REPO = "rajpurkarlab/ReXVQA"
REXGRAD_REPO = "rajpurkarlab/ReXGradient-160K"
DATA_FOLDER = "../data"

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import snapshot_download
import shutil
from pathlib import Path
import os

os.makedirs(DATA_FOLDER, exist_ok=True)
print("Downloading ReXVQA dataset...")

dest = Path(DATA_FOLDER) / "ReXVQA"
meta_path = snapshot_download(repo_id=REXVQA_REPO, repo_type="dataset", local_dir=dest)
print("ReXVQA dataset downloaded and moved successfully.")
print("Downloading ReXGradient-160K dataset...")

dest = Path(DATA_FOLDER) / "ReXGradient-160K"
meta_path = snapshot_download(repo_id=REXGRAD_REPO, repo_type="dataset", local_dir=dest)
shutil.move(meta_path, dest)
print("ReXGradient-160K dataset downloaded and moved successfully.")

In [ ]:
# Decompress the PNG archives for ReXGradient-160K
!cat {Path(DATA_FOLDER) / "ReXGradient-160K" / "deid_png.part*"} | tar --zstd -xf - -C  {Path(DATA_FOLDER) / "ReXGradient-160K"}

In [ ]:
# Move the png files to ReXVQA
src = Path(DATA_FOLDER) / "ReXGradient-160K" / "deid_png"
dest = Path(DATA_FOLDER) / "ReXVQA"
os.makedirs(dest, exist_ok=True)
shutil.move(src, dest)
print("PNG files moved to ReXVQA successfully.")

In [ ]:
# Now we can delete the ReXGradient-160K directory if we want to save space
shutil.rmtree(Path(DATA_FOLDER) / "ReXGradient-160K")
print("ReXGradient-160K directory removed successfully.")

In [ ]:
# Create mini validation set with 1% of the validation data with stratified sampling based on task_name
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

with open(Path(DATA_FOLDER) / "ReXVQA" / "metadata" / "valid_vqa_data.json", "r") as f:
    valid_data = json.load(f)
df = pd.DataFrame.from_dict(valid_data, orient="index")
_, mini_valid_df = train_test_split(
    df, test_size=0.01, stratify=df["task_name"], random_state=42
)
mini_valid_data = mini_valid_df.to_dict(orient="index")
print(f"Mini validation set created with {len(mini_valid_data)} samples.")
with open(
    Path(DATA_FOLDER) / "ReXVQA" / "metadata" / "valid_mini_vqa_data.json", "w"
) as f:
    json.dump(mini_valid_data, f)
print("Mini validation set created successfully.")